In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

# Day 3 Lab: Procedural Combinational Logic

> **Week 1, Session 3** · Accelerated HDL for Digital System Design · UCF ECE

## Overview

| | |
|---|---|
| **Duration** | ~2 hours |
| **Prerequisites** | Pre-class video (45 min): `always @(*)`, if/else, case, latch inference, blocking assignment |
| **Deliverable** | Mini-ALU with result on 7-segment, opcode selected by buttons |
| **Tools** | Yosys (critical for latch warnings!), nextpnr, iverilog |

## Learning Objectives

| SLO | Description |
|-----|-------------|
| 3.1 | Write `always @(*)` blocks for combinational logic |
| 3.2 | Implement decision structures (`if/else`, `case`, `casez`) and understand the hardware they imply |
| 3.3 | Identify and fix unintentional latch inference |
| 3.4 | Apply blocking assignment correctly in combinational blocks |
| 3.5 | Design a 4-bit ALU with procedural combinational logic |
| 3.6 | Use Yosys to detect latches in synthesized netlists |

## Exercises

### Exercise 1: Latch Hunting (20 min) ⚠️ MOST IMPORTANT
Find and fix intentional latch bugs. Run `make ex1_synth` and read every Yosys warning. Fix each bug in `starter/w1d3_ex1_latch_bugs.v`. Bug 3 is a trick question — explain why!

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex1_latch_bugs.v
// =============================================================================
// Exercise 1: Latch Hunting
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// BUGGY CODE — intentional latches for learning!
// Your job: synthesize, read the warnings, find and fix every latch.
//
// Steps:
//   1. Run: make ex1_synth
//   2. Read every Yosys warning — which signals have latches?
//   3. Fix each bug
//   4. Re-synthesize until all latch warnings are gone
// =============================================================================

module latch_bugs (
    input  wire [1:0] i_sel,
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [3:0] i_c,
    input  wire       i_enable,
    output reg  [3:0] o_result,
    output reg        o_flag,
    output reg  [2:0] o_encoded
);

    // --- Bug 1: Missing else ---
    // What happens to o_result when i_enable is 0?
    always @(*) begin
        if (i_enable)
            o_result = i_a + i_b;
        // TODO: Fix this — what value should o_result have when not enabled?
    end

    // --- Bug 2: Incomplete case ---
    // What happens when i_sel == 2'b11?
    always @(*) begin
        case (i_sel)
            2'b00: o_flag = 1'b0;
            2'b01: o_flag = 1'b1;
            2'b10: o_flag = 1'b0;
            // TODO: Fix — add the missing case or a default
        endcase
    end

    // --- Bug 3 (Trick Question!): Partial assignment in one branch ---
    // Is there a latch here? Explain why or why not.
    always @(*) begin
        o_encoded = 3'b000;          // default assignment at top
        case (i_sel)
            2'b00: o_encoded = 3'b001;
            2'b01: begin
                o_encoded[2] = 1'b1;
                // o_encoded[1:0] not explicitly assigned in this branch
                // But the default at top already assigned them...
            end
            2'b10: o_encoded = 3'b100;
            default: o_encoded = 3'b000;
        endcase
    end

endmodule

### Exercise 2: Priority Encoder (20 min)
Implement using `if/else` in `starter/w1d3_ex2_priority_encoder.v`. Compare with the provided `casez` alternative. Program and verify on board with `make ex2`.

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex2_priority_encoder.v
// =============================================================================
// Exercise 2: Priority Encoder
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// 4-input priority encoder
// Priority: bit 3 (highest) → bit 0 (lowest)
// =============================================================================

module priority_encoder (
    input  wire [3:0] i_request,
    output reg  [1:0] o_encoded,
    output reg        o_valid
);

    // Part A: Implement using if/else
    always @(*) begin
        // Default assignments — prevent latches
        o_encoded = 2'b00;
        o_valid   = 1'b0;

        // TODO: Use if/else chain
        // Check i_request[3] first (highest priority)
        // Set o_encoded to the bit position, o_valid to 1
        //
        // if (i_request[3]) begin
        //     ...
        // end else if (i_request[2]) begin
        //     ...
        // end ...

    end

endmodule

// Part B: Alternative implementation using casez (uncomment to test)
//
// module priority_encoder_casez (
//     input  wire [3:0] i_request,
//     output reg  [1:0] o_encoded,
//     output reg        o_valid
// );
//
//     always @(*) begin
//         o_encoded = 2'b00;
//         o_valid   = 1'b0;
//
//         casez (i_request)
//             4'b1???: begin o_encoded = 2'd3; o_valid = 1'b1; end
//             4'b01??: begin o_encoded = 2'd2; o_valid = 1'b1; end
//             4'b001?: begin o_encoded = 2'd1; o_valid = 1'b1; end
//             4'b0001: begin o_encoded = 2'd0; o_valid = 1'b1; end
//             default: begin o_encoded = 2'd0; o_valid = 1'b0; end
//         endcase
//     end
//
// endmodule

In [ ]:
%%writefile ex2_top_encoder.v
// =============================================================================
// Exercise 2: Top Module — Priority Encoder on Go Board
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module top_encoder (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,      // encoded[1]
    output wire o_led2,      // encoded[0]
    output wire o_led3,      // valid
    output wire o_led4       // unused
);

    wire [3:0] w_req = ~{i_switch1, i_switch2, i_switch3, i_switch4};
    wire [1:0] w_enc;
    wire       w_valid;

    priority_encoder enc (
        .i_request(w_req),
        .o_encoded(w_enc),
        .o_valid(w_valid)
    );

    assign o_led1 = ~w_enc[1];
    assign o_led2 = ~w_enc[0];
    assign o_led3 = ~w_valid;
    assign o_led4 = 1'b1;       // off

endmodule

### Exercise 3: 4-Bit ALU (35 min)
Four operations: ADD, SUB, AND, OR. Fill in `starter/w1d3_ex3_alu_4bit.v`. Fill in the verification matrix on paper before programming. Wire to board with `make ex3`.

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex3_alu_4bit.v
// =============================================================================
// Exercise 3: 4-Bit ALU
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Operations:
//   00 → ADD    01 → SUB    10 → AND    11 → OR
//
// Outputs:
//   o_result : 4-bit result
//   o_zero   : 1 if result is zero
//   o_carry  : carry/borrow output (ADD/SUB only)
// =============================================================================

module alu_4bit (
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_opcode,
    output reg  [3:0] o_result,
    output reg        o_zero,
    output reg        o_carry
);

    always @(*) begin
        // TODO: Default assignments (prevent latches on ALL outputs)

        case (i_opcode)
            2'b00: begin   // ADD
                // TODO: Use concatenation to capture carry
                // {o_carry, o_result} = i_a + i_b;
            end
            2'b01: begin   // SUB
                // TODO: Implement subtraction with borrow
                // {o_carry, o_result} = i_a - i_b;
            end
            2'b10: begin   // AND
                // TODO: Bitwise AND
            end
            2'b11: begin   // OR
                // TODO: Bitwise OR
            end
        endcase

        // TODO: Zero flag — use NOR reduction
        // o_zero = ~(|o_result);
    end

endmodule

In [ ]:
%%writefile ex3_top_alu.v
// =============================================================================
// Exercise 3: Top Module — ALU on Go Board
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// With only 4 switches, we can't fully control a + b + opcode.
// Design choice: Hardcode i_a = 4'd7, use sw1-sw2 as opcode, sw3-sw4 as i_b[1:0]
// =============================================================================

module top_alu (
    input  wire i_switch1,   // opcode[1]
    input  wire i_switch2,   // opcode[0]
    input  wire i_switch3,   // b[1]
    input  wire i_switch4,   // b[0]
    output wire o_led1,      // carry
    output wire o_led2,      // zero flag
    output wire o_led3,      // result[1]
    output wire o_led4,      // result[0]
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    wire [1:0] w_opcode = ~{i_switch1, i_switch2};
    wire [3:0] w_a = 4'd7;   // hardcoded operand a
    wire [3:0] w_b = {2'b00, ~i_switch3, ~i_switch4};

    wire [3:0] w_result;
    wire       w_zero, w_carry;

    // TODO: Instantiate alu_4bit
    // alu_4bit alu (
    //     .i_a(w_a), .i_b(w_b), .i_opcode(w_opcode),
    //     .o_result(w_result), .o_zero(w_zero), .o_carry(w_carry)
    // );

    // TODO: Drive LEDs (active-low) from carry, zero, result bits
    assign o_led1 = 1'b1;  // TODO: ~w_carry
    assign o_led2 = 1'b1;  // TODO: ~w_zero
    assign o_led3 = 1'b1;  // TODO: ~w_result[1]
    assign o_led4 = 1'b1;  // TODO: ~w_result[0]

    // TODO: Optionally drive 7-seg from result using hex_to_7seg from Day 2
    assign o_segment1_a = 1'b1;
    assign o_segment1_b = 1'b1;
    assign o_segment1_c = 1'b1;
    assign o_segment1_d = 1'b1;
    assign o_segment1_e = 1'b1;
    assign o_segment1_f = 1'b1;
    assign o_segment1_g = 1'b1;

endmodule

### Exercise 4: BCD-to-7-Seg Decoder (20 min)
Case-based decoder with error display. Compare readability with Day 2's nested conditional.

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex4_bcd_to_7seg.v
// =============================================================================
// Exercise 4: BCD-to-7-Segment Decoder
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Only valid for inputs 0–9. Inputs 10–15 display 'E' for error.
// Uses case statement — compare with Day 2's nested conditional approach.
// =============================================================================

module bcd_to_7seg (
    input  wire [3:0] i_bcd,
    output reg  [6:0] o_seg,
    output reg        o_valid
);

    always @(*) begin
        o_valid = 1'b1;
        o_seg   = 7'b1111111;  // default: all off

        case (i_bcd)
            4'd0:    o_seg = 7'b0000001;
            4'd1:    o_seg = 7'b1001111;
            4'd2:    o_seg = 7'b0010010;
            4'd3:    o_seg = 7'b0000110;
            4'd4:    o_seg = 7'b1001100;
            4'd5:    o_seg = 7'b0100100;
            4'd6:    o_seg = 7'b0100000;
            4'd7:    o_seg = 7'b0001111;
            4'd8:    o_seg = 7'b0000000;
            4'd9:    o_seg = 7'b0000100;
            default: begin
                o_seg   = 7'b0110000;  // 'E' for error
                o_valid = 1'b0;
            end
        endcase
    end

endmodule

In [ ]:
%%writefile ex4_top_bcd.v
// =============================================================================
// Exercise 4: Top Module — BCD Decoder on Go Board
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================

module top_bcd (
    input  wire i_switch1,
    input  wire i_switch2,
    input  wire i_switch3,
    input  wire i_switch4,
    output wire o_led1,      // valid indicator
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    wire [3:0] w_bcd = ~{i_switch1, i_switch2, i_switch3, i_switch4};
    wire [6:0] w_seg;
    wire       w_valid;

    bcd_to_7seg decoder (
        .i_bcd(w_bcd),
        .o_seg(w_seg),
        .o_valid(w_valid)
    );

    assign o_led1 = ~w_valid;  // LED on when valid BCD
    assign o_segment1_a = w_seg[6];
    assign o_segment1_b = w_seg[5];
    assign o_segment1_c = w_seg[4];
    assign o_segment1_d = w_seg[3];
    assign o_segment1_e = w_seg[2];
    assign o_segment1_f = w_seg[1];
    assign o_segment1_g = w_seg[0];

endmodule

### Exercise 5 — Stretch: ALU + 7-Seg Integration (25 min)
Full system: ALU result displayed on 7-seg, flags on LEDs.

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile ex5_top_alu_display.v
// =============================================================================
// Exercise 5 (Stretch): ALU + 7-Seg Integration
// Day 3 · Procedural Combinational Logic
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Hardcoded operands: a=7, b=3. Switches select opcode.
// Display ALU result on 7-seg, flags on LEDs.
// =============================================================================

module top_alu_display (
    input  wire i_switch1,   // opcode[1]
    input  wire i_switch2,   // opcode[0]
    output wire o_led1,      // carry
    output wire o_led2,      // zero
    output wire o_segment1_a,
    output wire o_segment1_b,
    output wire o_segment1_c,
    output wire o_segment1_d,
    output wire o_segment1_e,
    output wire o_segment1_f,
    output wire o_segment1_g
);

    // TODO:
    // 1. Wire opcode from switches
    // 2. Hardcode i_a = 4'd7, i_b = 4'd3
    // 3. Instantiate alu_4bit
    // 4. Instantiate hex_to_7seg (from Day 2) with the ALU result
    // 5. Drive LEDs from carry and zero flags
    // 6. Map segment outputs to pins

endmodule

## Deliverable Checklist

- [ ] Exercise 1: All latch warnings eliminated; Bug 3 explained
- [ ] Exercise 2: Priority encoder correct on board
- [ ] Exercise 3: ALU passes all verification matrix entries
- [ ] Exercise 4: BCD decoder shows 0-9, 'E' for 10-15
- [ ] At minimum: Exercise 3 (ALU) working on board

## Quick Reference

In [ ]:
!make ex1_synth    # Check for latch warnings

In [ ]:
!make ex2          make ex3          make ex4          make ex5

In [ ]:
!make clean